# 固定済み before / after ペアの部位別見た目解析

`video_before_after_pair.ipynb` で固定した `selected_pair.json` を使い、眉・上まぶた・唇・頬・額の記述的特徴量を比較します。

この notebook は **好印象・美しさ・若々しさの総合点を付けません**。Lab色、周囲皮膚とのコントラスト、頬の色ばらつき・明部率などを測り、人による印象評価と後で照合するための解析です。

入力画像・ROI・実装内容からfingerprintを作り、異なるrankの解析結果は別フォルダに保存します。既存結果を別入力として上書きしません。


In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
marker = Path('analysis/analyze_appearance.py')
if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('repo root:', Path.cwd())


## 設定


In [ ]:
VIDEO = Path('makeup0923.mp4')
OUTPUT = Path('outputs') / VIDEO.stem
SELECTED = OUTPUT / 'selected_pair.json'
APPEARANCE_ROOT = OUTPUT / 'appearance_selected_pair'

if not SELECTED.is_file():
    raise FileNotFoundError(
        f'固定済みペアがありません: {SELECTED}. '
        '先に video_before_after_pair.ipynb で候補を固定してください。'
    )

print('selected pair   :', SELECTED)
print('appearance root :', APPEARANCE_ROOT)


## 解析

元画像・ROI成果物のSHA-256を検証してから解析します。測定対象は元解像度の画素で、解析のためのリサイズや平滑化はしません。


In [ ]:
from analysis.analyze_appearance import analyze_selected_appearance, summary_html
from IPython.display import HTML, display

summary = analyze_selected_appearance(SELECTED, APPEARANCE_ROOT)
RUN_DIR = Path(summary['output_dir'])

print('appearance run:', RUN_DIR)
display(HTML(summary_html(summary, include_images=False)))


## 測定領域と変化を目視確認

自動値を読む前に、ROIが眉・上まぶた・唇・頬・額の意図した場所に入っているか、髪・手・強い影が混ざっていないか確認します。


In [ ]:
from IPython.display import Image as IPImage

for name in ('appearance_rois.png', 'region_samples.png', 'feature_changes.png'):
    path = RUN_DIR / name
    if not path.is_file():
        raise FileNotFoundError(path)
    print(name)
    display(IPImage(filename=str(path)))


## 生成ファイル


In [ ]:
for name in (
    'report.html', 'feature_summary.json', 'feature_deltas.csv',
    'appearance_rois.png', 'region_samples.png', 'feature_changes.png',
    'before_feature_masks.npz', 'after_feature_masks.npz',
):
    path = RUN_DIR / name
    if not path.is_file():
        raise FileNotFoundError(path)
    print(path)
